In [1]:
#import necessary libraries
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.processed.simulation import simulate_periodic_review_L0

In [2]:
demand_test = [20, 18, 25, 30, 15, 22, 19, 28, 24, 21, 17, 26, 20, 23]

results, summary = simulate_periodic_review_L0(
    mu_demand=10,
    std_demand=5,
    review_period=7,
    service_level=0.95,
    initial_inventory=20,
    demand_series=demand_test,
    holding_cost=1.5,
    shortage_cost=10,
    order_cost=50
)

print(results)
print(summary)

    period  demand  beginning_inventory  review_period_flag   z_value  \
0        1    20.0            20.000000                True  1.644854   
1        2    18.0            71.759368               False  1.644854   
2        3    25.0            53.759368               False  1.644854   
3        4    30.0            28.759368               False  1.644854   
4        5    15.0             0.000000               False  1.644854   
5        6    22.0             0.000000               False  1.644854   
6        7    19.0             0.000000               False  1.644854   
7        8    28.0             0.000000                True  1.644854   
8        9    24.0            63.759368               False  1.644854   
9       10    21.0            39.759368               False  1.644854   
10      11    17.0            18.759368               False  1.644854   
11      12    26.0             1.759368               False  1.644854   
12      13    20.0             0.000000            

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm


def simulate_periodic_review_L0(
    forecast_series,
    mu_error,
    std_error,
    review_period,
    service_level,
    initial_inventory,
    demand_series,
    holding_cost=0.0,
    shortage_cost=0.0,
    order_cost=0.0,
    return_details=True
):
    """
    Mô phỏng hệ thống tồn kho periodic review với:
    - Lead time L = 0
    - Safety stock:
        SS = z_alpha * sigma_E * sqrt(R)
    - Order-up-to level:
        S = D_hat_R + mu_E * R + SS

    Parameters
    ----------
    forecast_series : list, np.ndarray, pd.Series
        Chuỗi forecast theo từng kỳ.
    mu_error : float
        Trung bình sai số forecast mỗi kỳ.
    std_error : float
        Độ lệch chuẩn sai số forecast mỗi kỳ.
    review_period : int
        Chu kỳ kiểm tra tồn kho R.
    service_level : float
        Mức độ phục vụ mong muốn, ví dụ 0.95.
    initial_inventory : float
        Tồn kho đầu kỳ.
    demand_series : list, np.ndarray, pd.Series
        Chuỗi nhu cầu thực tế dùng để chạy mô phỏng.
    holding_cost : float, default=0.0
        Chi phí lưu kho trên mỗi đơn vị tồn cuối kỳ.
    shortage_cost : float, default=0.0
        Chi phí thiếu hàng trên mỗi đơn vị thiếu.
    order_cost : float, default=0.0
        Chi phí cố định cho mỗi lần đặt hàng.
    return_details : bool, default=True
        Nếu True trả về bảng chi tiết từng kỳ.

    Returns
    -------
    results_df : pd.DataFrame or None
        Bảng mô phỏng chi tiết theo từng kỳ.
    summary : dict
        Thống kê tổng hợp mô phỏng.
    """

    # =========================
    # 0. VALIDATION
    # =========================
    if review_period <= 0:
        raise ValueError("review_period phải > 0")
    if std_error < 0:
        raise ValueError("std_error không được âm")
    if not (0 < service_level < 1):
        raise ValueError("service_level phải nằm trong khoảng (0, 1)")
    if initial_inventory < 0:
        raise ValueError("initial_inventory không được âm")

    demand_series = pd.Series(demand_series, dtype=float).reset_index(drop=True)
    forecast_series = pd.Series(forecast_series, dtype=float).reset_index(drop=True)

    if len(demand_series) == 0:
        raise ValueError("demand_series không được rỗng")
    if len(forecast_series) == 0:
        raise ValueError("forecast_series không được rỗng")
    if len(forecast_series) < len(demand_series):
        raise ValueError("forecast_series phải có độ dài >= demand_series")

    # =========================
    # 1. TÍNH POLICY THEO LÝ THUYẾT
    # =========================
    z = norm.ppf(service_level)

    # Safety stock cố định theo công thức:
    # SS = z_alpha * sigma_E * sqrt(R)
    safety_stock = z * std_error * np.sqrt(review_period)

    # =========================
    # 2. MÔ PHỎNG
    # =========================
    inventory = float(initial_inventory)

    total_holding_cost = 0.0
    total_shortage_cost = 0.0
    total_order_cost = 0.0
    total_order_qty = 0.0
    total_shortage_units = 0.0
    total_sales = 0.0
    num_orders = 0

    records = []

    for t, demand in enumerate(demand_series, start=1):
        beginning_inventory = inventory
        is_review_period = ((t - 1) % review_period == 0)

        order_qty = 0.0
        expected_demand_over_review = np.nan
        bias_correction = np.nan
        order_up_to_level = np.nan

        # Review đầu kỳ; do L=0 nên nhận ngay
        if is_review_period:
            start_idx = t - 1
            end_idx = min(start_idx + review_period, len(forecast_series))

            # Tổng forecast trong kỳ review
            forecast_window = forecast_series.iloc[start_idx:end_idx]
            expected_demand_over_review = forecast_window.sum()

            # Bias correction = mu_E * R
            bias_correction = mu_error * review_period

            # Order-up-to level:
            # S = D_hat_R + mu_E * R + SS
            order_up_to_level = (
                expected_demand_over_review
                + bias_correction
                + safety_stock
            )

            inventory_position = inventory

            if inventory_position < order_up_to_level:
                order_qty = order_up_to_level - inventory_position
                inventory += order_qty
                total_order_qty += order_qty
                num_orders += 1
                if order_qty > 0:
                    total_order_cost += order_cost

        inventory_after_replenishment = inventory

        # Nhu cầu xảy ra trong kỳ
        sales = min(inventory, demand)
        shortage = max(demand - inventory, 0.0)
        ending_inventory = max(inventory - demand, 0.0)

        # Chi phí
        period_holding_cost = ending_inventory * holding_cost
        period_shortage_cost = shortage * shortage_cost

        # Cập nhật tổng
        total_sales += sales
        total_shortage_units += shortage
        total_holding_cost += period_holding_cost
        total_shortage_cost += period_shortage_cost

        # Tồn kho chuyển sang kỳ sau
        inventory = ending_inventory

        if return_details:
            current_forecast = forecast_series.iloc[t - 1]
            forecast_error = demand - current_forecast

            records.append({
                "period": t,
                "forecast": current_forecast,
                "demand": demand,
                "forecast_error": forecast_error,
                "beginning_inventory": beginning_inventory,
                "review_period_flag": is_review_period,
                "z_value": z,
                "mu_error": mu_error,
                "std_error": std_error,
                "safety_stock": safety_stock,
                "forecast_sum_R": expected_demand_over_review,
                "bias_correction": bias_correction,
                "order_up_to_level": order_up_to_level,
                "order_qty": order_qty,
                "inventory_after_replenishment": inventory_after_replenishment,
                "sales": sales,
                "shortage": shortage,
                "ending_inventory": ending_inventory,
                "holding_cost": period_holding_cost,
                "shortage_cost": period_shortage_cost
            })

    # =========================
    # 3. XUẤT DATAFRAME KẾT QUẢ
    # =========================
    results_df = pd.DataFrame(records) if return_details else None

    # =========================
    # 4. TỔNG HỢP KẾT QUẢ
    # =========================
    total_demand = demand_series.sum()
    fill_rate = total_sales / total_demand if total_demand > 0 else 1.0

    if return_details and len(results_df) > 0:
        daily_service_level_empirical = (results_df["shortage"] == 0).mean()
    else:
        daily_service_level_empirical = None

    # Cycle service level thực nghiệm
    stockout_cycles = 0
    total_cycles = 0

    if return_details and len(results_df) > 0:
        for start_idx in range(0, len(demand_series), review_period):
            end_idx = min(start_idx + review_period, len(demand_series))
            cycle_df = results_df.iloc[start_idx:end_idx]
            total_cycles += 1
            if cycle_df["shortage"].sum() > 0:
                stockout_cycles += 1

    cycle_service_level_empirical = (
        1 - stockout_cycles / total_cycles if total_cycles > 0 else 1.0
    )

    summary = {
        "review_period": review_period,
        "lead_time": 0,
        "service_level_target": service_level,
        "z_value": z,
        "mu_error": mu_error,
        "std_error": std_error,
        "safety_stock": safety_stock,
        "initial_inventory": initial_inventory,
        "num_periods_simulated": len(demand_series),
        "num_orders": num_orders,
        "total_demand": total_demand,
        "total_sales": total_sales,
        "total_shortage_units": total_shortage_units,
        "fill_rate": fill_rate,
        "cycle_service_level_empirical": cycle_service_level_empirical,
        "daily_service_level_empirical": daily_service_level_empirical,
        "average_ending_inventory": (
            results_df["ending_inventory"].mean()
            if return_details and len(results_df) > 0 else None
        ),
        "total_order_qty": total_order_qty,
        "total_holding_cost": total_holding_cost,
        "total_shortage_cost": total_shortage_cost,
        "total_order_cost": total_order_cost,
        "total_cost": total_holding_cost + total_shortage_cost + total_order_cost
    }

    return results_df, summary

In [3]:
forecast_series = [30, 20, 60, 90, 50, 40, 70, 80]
demand_series   = [98, 120, 90, 100, 130, 110, 95, 108]

results_df, summary = simulate_periodic_review_L0(
    forecast_series=forecast_series,
    mu_error=2.0,
    std_error=10.0,
    review_period=4,
    service_level=0.95,
    initial_inventory=120,
    demand_series=demand_series,
    holding_cost=1.0,
    shortage_cost=5.0,
    order_cost=20.0,
    return_details=True
)

print(results_df)
print(summary)

   period  forecast  demand  forecast_error  beginning_inventory  \
0       1      30.0    98.0            68.0           120.000000   
1       2      20.0   120.0           100.0           142.897073   
2       3      60.0    90.0            30.0            22.897073   
3       4      90.0   100.0            10.0             0.000000   
4       5      50.0   130.0            80.0             0.000000   
5       6      40.0   110.0            70.0           150.897073   
6       7      70.0    95.0            25.0            40.897073   
7       8      80.0   108.0            28.0             0.000000   

   review_period_flag   z_value  mu_error  std_error  safety_stock  \
0                True  1.644854       2.0       10.0     32.897073   
1               False  1.644854       2.0       10.0     32.897073   
2               False  1.644854       2.0       10.0     32.897073   
3               False  1.644854       2.0       10.0     32.897073   
4                True  1.644854      